# Pix2Pix Dataset Preparation — v3

**Patch filter logic (decision made on Trichrome channel):**
1. ImageJ-style histogram stretch (linear, 0.35% saturation per tail)
2. Build black pixel mask: pixels where ALL channels < BLACK_THRESHOLD
3. Run masked Otsu independently on R, G, B (ignoring black pixels)
4. Combine tissue maps with OR: pixel is tissue if ANY channel says tissue
5. Tile the combined mask: keep patch if >= 50% of pixels are tissue
6. Remove small isolated tissue islands (< MIN_COMPONENT_SIZE patches)

**Split:** 80% train / 10% val / 10% test (pooled across all samples)

**Tasks:** HE->PAS (samples 1-4, 8)  |  HE->TRI (samples 1-8)


In [ ]:
import os
import numpy as np
import tifffile as tif
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration

In [ ]:
OUTPUT_ROOT = r'' # path to save

PATCH_SIZE   = 512
TRAIN_RATIO  = 0.80
RANDOM_STATE = 42

# ImageJ Enhance Contrast saturation fraction
SATURATE_FRAC = 0.0035

# Pixels where ALL channels < this are black artefacts and ignored by Otsu
BLACK_THRESHOLD = 20

# Connected component filter
MIN_COMPONENT_SIZE = 10

VIZ_DOWNSAMPLE = 8

SAMPLES = [
    {'id': 'sample_1',    'path': r'',  'has_pas': True},
]

## 2. Helper Functions

In [ ]:
def create_pix2pix_folders(base):
    for domain in ('A', 'B'):
        for split in ('train', 'val', 'test'):
            os.makedirs(os.path.join(base, domain, split), exist_ok=True)


def load_tif(path):
    img = tif.imread(path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[-1] > 3:
        img = img[..., :3]
    if img.dtype == np.uint16:
        img = (img / 65535.0 * 255).astype(np.uint8)
    elif img.dtype != np.uint8:
        img = ((img - img.min()) / (img.max() - img.min() + 1e-8) * 255).astype(np.uint8)
    return img


def save_png(img, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


def tile_image(img, patch_size):
    H, W   = img.shape[:2]
    H_trim = (H // patch_size) * patch_size
    W_trim = (W // patch_size) * patch_size
    img    = img[:H_trim, :W_trim]
    C      = img.shape[2]
    n_rows = H_trim // patch_size
    n_cols = W_trim // patch_size
    patches = (
        img
        .reshape(n_rows, patch_size, n_cols, patch_size, C)
        .swapaxes(1, 2)
        .reshape(-1, patch_size, patch_size, C)
    )
    return patches, (n_rows, n_cols)


# ── ImageJ contrast enhancement ────────────────────────────────────────────────

def enhance_contrast_imagej(img, saturate_frac=SATURATE_FRAC):
    out      = np.empty_like(img)
    n_pixels = img.shape[0] * img.shape[1]
    n_sat    = int(n_pixels * saturate_frac)
    for c in range(img.shape[2]):
        channel = img[:, :, c]
        hist    = np.bincount(channel.ravel(), minlength=256)
        cum     = np.cumsum(hist)
        low     = int(np.searchsorted(cum, n_sat))
        high    = int(np.searchsorted(cum, n_pixels - n_sat))
        if high <= low:
            out[:, :, c] = channel
            continue
        stretched = (channel.astype(np.float32) - low) / (high - low) * 255.0
        out[:, :, c] = np.clip(stretched, 0, 255).astype(np.uint8)
    return out


# ── Masked Otsu ────────────────────────────────────────────────────────────────

def otsu_masked(channel, black_mask):
    """
    Compute Otsu threshold on a single channel ignoring black pixels.
    Returns thresh and tissue_map (True = tissue = below Otsu threshold).
    Black pixels are excluded from both the threshold calculation
    and the output tissue map.
    """
    valid_pixels = channel[~black_mask]
    if len(valid_pixels) == 0:
        return 0, np.zeros(channel.shape, dtype=bool)

    hist  = np.bincount(valid_pixels, minlength=256).astype(np.float64)
    total = hist.sum()
    sum_total = np.dot(np.arange(256), hist)
    sum_b, w_b, max_var, thresh = 0.0, 0.0, 0.0, 0

    for t in range(256):
        w_b += hist[t]
        if w_b == 0: continue
        w_f = total - w_b
        if w_f == 0: break
        sum_b  += t * hist[t]
        mean_b  = sum_b / w_b
        mean_f  = (sum_total - sum_b) / w_f
        var     = w_b * w_f * (mean_b - mean_f) ** 2
        if var > max_var:
            max_var = var
            thresh  = t

    tissue_map = (channel <= thresh) & ~black_mask
    return thresh, tissue_map


# ── Combined tissue mask ───────────────────────────────────────────────────────

def build_tissue_mask(tri_enhanced, black_threshold=BLACK_THRESHOLD):
    """
    1. Black mask: ALL channels < black_threshold
    2. Masked Otsu on R, G, B independently
    3. OR combination: tissue if any channel says tissue
    Returns combined tissue mask (bool), black mask (bool), thresholds.
    """
    R = tri_enhanced[:, :, 0]
    G = tri_enhanced[:, :, 1]
    B = tri_enhanced[:, :, 2]

    black_mask = (R < black_threshold) & (G < black_threshold) & (B < black_threshold)

    tr, tissue_r = otsu_masked(R, black_mask)
    tg, tissue_g = otsu_masked(G, black_mask)
    tb, tissue_b = otsu_masked(B, black_mask)

    combined = tissue_r | tissue_g | tissue_b
    return combined, black_mask, (tr, tg, tb)


# ── Tile boolean mask ──────────────────────────────────────────────────────────

def tile_bool_mask(mask_2d, patch_size):
    """Tile a 2D bool mask. Patch is kept if >= 50% pixels are True."""
    H, W   = mask_2d.shape
    H_trim = (H // patch_size) * patch_size
    W_trim = (W // patch_size) * patch_size
    mask   = mask_2d[:H_trim, :W_trim]
    n_rows = H_trim // patch_size
    n_cols = W_trim // patch_size
    patches = (
        mask
        .reshape(n_rows, patch_size, n_cols, patch_size)
        .swapaxes(1, 2)
        .reshape(-1, patch_size, patch_size)
    )
    kept_mask = patches.mean(axis=(1, 2)) >= 0.5
    return kept_mask, n_rows, n_cols


# ── Connected-component filter ─────────────────────────────────────────────────

def filter_small_islands(kept_mask, n_rows, n_cols, min_size=MIN_COMPONENT_SIZE):
    tissue_grid = kept_mask.reshape(n_rows, n_cols).astype(np.int32)
    structure   = np.ones((3, 3), dtype=np.int32)
    comp_grid, n_comp = ndimage.label(tissue_grid, structure=structure)
    comp_sizes  = np.bincount(comp_grid.ravel())
    small_ids   = {cid for cid in range(1, n_comp + 1) if comp_sizes[cid] < min_size}
    small_mask  = np.array([comp_grid.ravel()[i] in small_ids
                            for i in range(len(kept_mask))])
    kept_mask   = kept_mask & ~small_mask
    print(f"  CC filter: {n_comp} components, "
          f"{len(small_ids)} too small (<{min_size}), "
          f"{small_mask.sum()} patches removed")
    return kept_mask, small_mask


# ── Train/val/test split ───────────────────────────────────────────────────────

def split_indices(n):
    idx = np.arange(n)
    train_idx, temp_idx = train_test_split(
        idx, test_size=(1 - TRAIN_RATIO), random_state=RANDOM_STATE, shuffle=True)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, random_state=RANDOM_STATE, shuffle=True)
    return train_idx, val_idx, test_idx


# ── Visualization ──────────────────────────────────────────────────────────────

def draw_boxes(img, kept_mask, small_mask, n_rows, n_cols, patch_size, downsample):
    H_trim = n_rows * patch_size
    W_trim = n_cols * patch_size
    small  = img[:H_trim, :W_trim][::downsample, ::downsample].copy()
    ps     = patch_size // downsample
    thick  = max(1, ps // 20)
    for idx in range(len(kept_mask)):
        r, c   = divmod(idx, n_cols)
        y1, x1 = r * ps, c * ps
        y2, x2 = y1 + ps, x1 + ps
        if kept_mask[idx]:
            color, alpha, border = (220, 30, 30), 0.45, True
        elif small_mask[idx]:
            color, alpha, border = (140, 140, 140), 0.50, False
        else:
            color, alpha, border = (30, 80, 220), 0.25, False
        small[y1:y2, x1:x2] = (
            small[y1:y2, x1:x2].astype(np.float32) * (1 - alpha)
            + np.array(color, dtype=np.float32) * alpha
        ).clip(0, 255).astype(np.uint8)
        if border:
            cv2.rectangle(small, (x1, y1), (x2-1, y2-1), color, thick)
    return small


def visualize_sample(sid, he, pas, tri, kept_mask, small_mask,
                     n_rows, n_cols, thresholds):
    n_total  = len(kept_mask)
    n_kept   = kept_mask.sum()
    n_island = small_mask.sum()
    n_bg     = n_total - n_kept - n_island
    tr, tg, tb = thresholds

    fig, axes = plt.subplots(1, 3, figsize=(19, 7))
    fig.suptitle(
        f"{sid}\n"
        f"Total: {n_total}  |  "
        f"Tissue (kept): {n_kept} ({100*n_kept/n_total:.1f}%)  |  "
        f"Background: {n_bg} ({100*n_bg/n_total:.1f}%)  |  "
        f"Small island: {n_island} ({100*n_island/n_total:.1f}%)\n"
        f"ImageJ stretch ({SATURATE_FRAC*100:.2f}%)  |  "
        f"Masked Otsu OR — R={tr} G={tg} B={tb}  |  "
        f"BLACK_THRESHOLD={BLACK_THRESHOLD}  |  MIN_COMPONENT_SIZE={MIN_COMPONENT_SIZE}",
        fontsize=9, y=1.03,
    )
    for ax, (stain_label, img) in zip(axes, [('HE', he), ('PAS', pas), ('Trichrome', tri)]):
        ax.set_title(stain_label, fontsize=13, fontweight='bold')
        ax.axis('off')
        if img is None:
            ph = np.full((200, 200, 3), 180, dtype=np.uint8)
            ax.imshow(ph)
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='#555')
        else:
            ax.imshow(draw_boxes(img, kept_mask, small_mask,
                                 n_rows, n_cols, PATCH_SIZE, VIZ_DOWNSAMPLE))
    legend_handles = [
        mpatches.Patch(color=(220/255,  30/255,  30/255), label=f'Tissue — kept ({n_kept})'),
        mpatches.Patch(color=( 30/255,  80/255, 220/255), label=f'Background — discarded ({n_bg})'),
        mpatches.Patch(color=(140/255, 140/255, 140/255), label=f'Small island — discarded ({n_island})'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=3,
               fontsize=10, framealpha=0.85, bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout()
    plt.show()

## 3. Per-Sample Processing + Visualization

In [ ]:
all_pas_pairs = []
all_tri_pairs = []

for sample in SAMPLES:
    sid, spath, has_pas = sample['id'], sample['path'], sample['has_pas']
    print(f"\n{'='*60}")
    print(f"Processing {sid}  |  PAS: {has_pas}")

    try:
        he  = load_tif(os.path.join(spath, 'HE_registered.tif'))
        tri = load_tif(os.path.join(spath, 'Trichrome_registered.tif'))
        pas = load_tif(os.path.join(spath, 'PAS_registered.tif')) if has_pas else None
    except Exception as e:
        print(f"  ERROR: {e}  -> Skipping.")
        continue

    print(f"  HE {he.shape}  TRI {tri.shape}" +
          (f"  PAS {pas.shape}" if pas is not None else "  PAS N/A"))

    if pas is not None and he.shape[:2] != pas.shape[:2]:
        print("  WARNING: HE/PAS mismatch -> dropping PAS")
        pas = None
    if he.shape[:2] != tri.shape[:2]:
        print("  WARNING: HE/TRI mismatch -> dropping TRI")
        tri = None

    if tri is None:
        print("  No Trichrome -> skipping sample")
        continue

    # Step 1: ImageJ contrast enhancement
    tri_enhanced = enhance_contrast_imagej(tri)

    # Step 2+3: Masked Otsu on R, G, B + OR combination
    combined_tissue, black_mask, thresholds = build_tissue_mask(tri_enhanced)
    print(f"  Otsu thresholds — R:{thresholds[0]} G:{thresholds[1]} B:{thresholds[2]}  "
          f"| Black pixels: {black_mask.mean()*100:.1f}%  "
          f"| Tissue pixels: {combined_tissue.mean()*100:.1f}%")

    # Step 4: Tile the combined mask
    kept_mask, n_rows, n_cols = tile_bool_mask(combined_tissue, PATCH_SIZE)
    print(f"  Initial tissue patches: {kept_mask.sum()} / {len(kept_mask)}")

    # Step 5: Remove small isolated islands
    kept_mask, small_mask = filter_small_islands(kept_mask, n_rows, n_cols)

    n_tissue = kept_mask.sum()
    n_total  = len(kept_mask)
    print(f"  Final tissue patches: {n_tissue} / {n_total} ({100*n_tissue/n_total:.1f}%)")

    visualize_sample(sid, he, pas, tri, kept_mask, small_mask,
                     n_rows, n_cols, thresholds)

    # Tile and accumulate
    he_patches,  _ = tile_image(he,  PATCH_SIZE)
    tri_patches, _ = tile_image(tri, PATCH_SIZE)
    pas_patches, _ = tile_image(pas, PATCH_SIZE) if pas is not None else (None, None)

    for i in np.where(kept_mask)[0]:
        he_p = he_patches[i].copy()
        if pas_patches is not None:
            all_pas_pairs.append((he_p, pas_patches[i].copy()))
        all_tri_pairs.append((he_p, tri_patches[i].copy()))

print(f"\n{'='*60}")
print(f"Total HE->PAS pairs : {len(all_pas_pairs)}")
print(f"Total HE->TRI pairs : {len(all_tri_pairs)}")

## 4. Split and Save

In [ ]:
def save_dataset(pairs, dataset_name):
    base = os.path.join(OUTPUT_ROOT, dataset_name)
    create_pix2pix_folders(base)
    n = len(pairs)
    train_idx, val_idx, test_idx = split_indices(n)
    print(f"\n{dataset_name}  total={n}  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
    for split_name, indices in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
        for new_i, orig_i in enumerate(tqdm(indices, desc=f"  {split_name}")):
            a, b = pairs[orig_i]
            save_png(a, os.path.join(base, 'A', split_name, f"{new_i}.png"))
            save_png(b, os.path.join(base, 'B', split_name, f"{new_i}.png"))
    print(f"  Saved -> {base}")

save_dataset(all_pas_pairs, 'HE_PAS')
save_dataset(all_tri_pairs, 'HE_TRI')

## 5. Sanity Check

In [ ]:
def sanity_check(dataset_name):
    base = os.path.join(OUTPUT_ROOT, dataset_name)
    print(f"\n{dataset_name}")
    total_a = total_b = 0
    for split in ('train', 'val', 'test'):
        a = sorted(os.listdir(os.path.join(base, 'A', split)))
        b = sorted(os.listdir(os.path.join(base, 'B', split)))
        ok = 'OK' if a == b else 'MISMATCH'
        print(f"  {split:6s}  A:{len(a):5d}  B:{len(b):5d}  {ok}")
        total_a += len(a); total_b += len(b)
    print(f"  {'TOTAL':6s}  A:{total_a:5d}  B:{total_b:5d}")

sanity_check('HE_PAS')
sanity_check('HE_TRI')

## 6. Build AB Folder (pix2pix `--dataset_mode aligned`)

In [ ]:
def make_AB_folder(dataset_name):
    base = os.path.join(OUTPUT_ROOT, dataset_name)
    print(f"\nBuilding AB: {dataset_name}")
    for split in ('train', 'val', 'test'):
        a_dir  = os.path.join(base, 'A', split)
        b_dir  = os.path.join(base, 'B', split)
        ab_dir = os.path.join(base, 'AB', split)
        os.makedirs(ab_dir, exist_ok=True)
        files = sorted(os.listdir(a_dir))
        for fname in tqdm(files, desc=f"  {split}"):
            a_img = cv2.imread(os.path.join(a_dir, fname))
            b_img = cv2.imread(os.path.join(b_dir, fname))
            cv2.imwrite(os.path.join(ab_dir, fname),
                        np.concatenate([a_img, b_img], axis=1))
        print(f"  {split}: {len(files)} images")
    print(f"  Done -> {os.path.join(base, 'AB')}")

make_AB_folder('HE_PAS')
make_AB_folder('HE_TRI')